In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import sys
ROOT = Path.cwd().parent.parent
sys.path.append(str(ROOT))
from src.loading_data.load_data import get_clean_2022
from src.perturbation.fitting_models import preprocess_perturbation
from src.loading_data.data_catalogue import DataCatalogue
from sklearn.preprocessing import StandardScaler
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score

c:\Users\fergu\Documents\GitHub\london_sport2\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = get_clean_2022()

['Gend3', 'Disab2_POP', 'Age9', 'Eth7', 'NSSEC5', 'Educ6', 'IMD10', 'Child4', 'WorkStat8', 'HHLiv9', 'serial', 'year', 'LCA_Class', 'MEMS7_ALL']
Data loaded with zero missing values.
Loaded primary frame...
Loaded secondary frame...
Merged Frames.
Dropped NAN values
Dropped: ['serial', 'year', 'MEMS7_ALL]


In [3]:
df.columns

Index(['LCA_Class', 'Age9', 'Gend3', 'Eth7', 'Disab2_POP', 'Educ6', 'NSSEC5',
       'IMD10', 'WorkStat8', 'Child4', 'HHLiv9', 'active', 'Motiva_POP',
       'motivd_POP', 'inclus_a', 'inclus_b', 'inclus_c', 'anxious', 'comm1',
       'comm2', 'happy', 'indev', 'indevtry', 'lone', 'worthw'],
      dtype='str')

In [4]:
dc = DataCatalogue()
categoricals = dc.get_perturbation_processing_categoricals()
continuous_vars = dc.get_perturbation_core_contins() + dc.get_perturbation_vars()
print(categoricals)

['active', 'Gend3', 'Disab2_POP', 'Eth7', 'WorkStat8', 'HHLiv9']


In [ ]:
X_train, X_test, Y_train, Y_test, test_set_clusters = preprocess_perturbation(df, 'LCA_Class', 'active', categoricals, save = False)

Keep columns : ['Age9', 'Gend3', 'Eth7', 'Disab2_POP', 'Educ6', 'NSSEC5', 'IMD10', 'WorkStat8', 'Child4', 'HHLiv9', 'Motiva_POP', 'motivd_POP', 'inclus_a', 'inclus_b', 'inclus_c', 'anxious', 'comm1', 'comm2', 'happy', 'indev', 'indevtry', 'lone', 'worthw']
X_train cols:
Index(['Age9', 'Gend3', 'Eth7', 'Disab2_POP', 'Educ6', 'NSSEC5', 'IMD10',
       'WorkStat8', 'Child4', 'HHLiv9', 'Motiva_POP', 'motivd_POP', 'inclus_a',
       'inclus_b', 'inclus_c', 'anxious', 'comm1', 'comm2', 'happy', 'indev',
       'indevtry', 'lone', 'worthw'],
      dtype='str')
X_train shape:
(2628, 23)
X_train values:
       Age9 Gend3 Eth7 Disab2_POP  Educ6  NSSEC5  IMD10 WorkStat8  Child4  \
9825    4.0     1    1          1    1.0     1.0   10.0         0     4.0   
4034    8.0     1    1          1    1.0     5.0    8.0         3     1.0   
189     4.0     1    3          1    1.0     1.0    5.0         0     2.0   
2050    4.0     2    1          1    1.0     1.0    8.0         1     2.0   
12453   3.0  

In [ ]:
scaler = StandardScaler()
X_train[continuous_vars] = scaler.fit_transform(X_train[continuous_vars])

X_test[continuous_vars] = scaler.transform(X_test[continuous_vars])

In [ ]:
X_train

In [7]:
model1 = LGBMClassifier(random_state = 42,verbose = -1, class_weight='balanced')
model1.fit(X_train, Y_train)
preds = model1.predict(X_test)
f1 = f1_score(Y_test, preds, average='macro')
print(f1)

0.6122590780485517


In [8]:
model1.get_params()

{'boosting_type': 'gbdt',
 'class_weight': 'balanced',
 'colsample_bytree': 1.0,
 'importance_type': 'split',
 'learning_rate': 0.1,
 'max_depth': -1,
 'min_child_samples': 20,
 'min_child_weight': 0.001,
 'min_split_gain': 0.0,
 'n_estimators': 100,
 'n_jobs': None,
 'num_leaves': 31,
 'objective': None,
 'random_state': 42,
 'reg_alpha': 0.0,
 'reg_lambda': 0.0,
 'subsample': 1.0,
 'subsample_for_bin': 200000,
 'subsample_freq': 0,
 'verbose': -1}

In [9]:
model2 = XGBClassifier(random_state = 42)
model2.fit(X_train, Y_train)
preds = model2.predict(X_test)
f1 = f1_score(Y_test, preds, average='macro')
print(f1)


0.5811230585424134


In [10]:
model3 = RandomForestClassifier(random_state=42, class_weight='balanced')
model3.fit(X_train, Y_train)
preds = model3.predict(X_test)
f1 = f1_score(Y_test, preds, average='macro')
print(f1)

0.6083516670930331


In [11]:
model4 = CatBoostClassifier(random_state = 42, auto_class_weights='Balanced', iterations=30) # auto_class_weights='balanced'
model4.fit(X_train, Y_train, cat_features=['Gend3', 'Disab2_POP', 'Eth7', 'WorkStat8', 'HHLiv9'])
preds = model4.predict(X_test)
f1 = f1_score(Y_test, preds, average='macro')
print(f1)

Learning rate set to 0.387788
0:	learn: 0.6598287	total: 196ms	remaining: 5.69s
1:	learn: 0.6442375	total: 238ms	remaining: 3.34s
2:	learn: 0.6290993	total: 290ms	remaining: 2.61s
3:	learn: 0.6149264	total: 333ms	remaining: 2.17s
4:	learn: 0.6046573	total: 382ms	remaining: 1.91s
5:	learn: 0.5901906	total: 430ms	remaining: 1.72s
6:	learn: 0.5827557	total: 481ms	remaining: 1.58s
7:	learn: 0.5779437	total: 529ms	remaining: 1.45s
8:	learn: 0.5709742	total: 579ms	remaining: 1.35s
9:	learn: 0.5670644	total: 629ms	remaining: 1.26s
10:	learn: 0.5621580	total: 681ms	remaining: 1.18s
11:	learn: 0.5582940	total: 735ms	remaining: 1.1s
12:	learn: 0.5544404	total: 788ms	remaining: 1.03s
13:	learn: 0.5533058	total: 825ms	remaining: 943ms
14:	learn: 0.5481381	total: 876ms	remaining: 876ms
15:	learn: 0.5423017	total: 927ms	remaining: 811ms
16:	learn: 0.5338931	total: 982ms	remaining: 751ms
17:	learn: 0.5264355	total: 1.04s	remaining: 691ms
18:	learn: 0.5222335	total: 1.09s	remaining: 631ms
19:	learn: 0

In [12]:
Y_train.value_counts()[0]

np.int64(707)